# 오류 분석: 실패 케이스 분석

핵심: 틀린 것을 직시하고 원인을 찾는다

## 분석 대상
- 모델: TensorFlow EfficientNetV2S (실험 결과 채택 모델)
- 데이터: roboflow 공개 데이터셋 8개 클래스
- test accuracy: 70.5%

In [ ]:
# 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 라이브러리 import
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# 경로 및 클래스 설정
MERGED = '/content/drive/MyDrive/해커톤/dataset/merged'
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
class_names = ['TV', 'air_conditioning', 'fridge', 'pet_bottle',
               'chair', 'ring', 'paper_document', 'album']
NUM_CLASSES = len(class_names)

In [ ]:
# 데이터 로드
def load_dataset(split):
    images, labels = [], []
    for idx, cls in enumerate(class_names):
        img_dir = f"{MERGED}/{split}/images"
        for f in os.listdir(img_dir):
            if f.startswith(cls + '_'):
                img_path = os.path.join(img_dir, f)
                img = tf.keras.utils.load_img(img_path, target_size=IMG_SIZE)
                images.append(tf.keras.utils.img_to_array(img) / 255.0)
                labels.append(idx)
    return np.array(images), tf.keras.utils.to_categorical(labels, NUM_CLASSES)

print('데이터 로드 중...')
X_train, y_train = load_dataset('train')
X_val,   y_val   = load_dataset('valid')
X_test,  y_test  = load_dataset('test')
print(f'train: {len(X_train)}, val: {len(X_val)}, test: {len(X_test)}')

In [ ]:
# 모델 학습 (3_experiments.ipynb에서 이미 학습한 경우 생략 가능)
base_model = EfficientNetV2S(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

inputs  = tf.keras.Input(shape=(224, 224, 3))
x       = base_model(inputs, training=False)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.BatchNormalization()(x)
x       = layers.Dropout(0.5)(x)
x       = layers.Dense(256, activation='relu')(x)
x       = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True, monitor='val_accuracy'),
    ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-6),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=BATCH_SIZE,
    callbacks=callbacks
)

In [ ]:
# ① Confusion Matrix (혼동 행렬)
# test 데이터 전체 예측
y_pred_prob = model.predict(X_test, verbose=0)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = np.argmax(y_test, axis=1)

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/해커톤/confusion_matrix.png', dpi=100)
plt.show()

print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
# ② 가장 자주 혼동되는 클래스 쌍 Top 10
confusion_pairs = []
for i in range(len(class_names)):
    for j in range(len(class_names)):
        if i != j and cm[i][j] > 0:
            confusion_pairs.append({
                'actual':    class_names[i],
                'predicted': class_names[j],
                'count':     cm[i][j]
            })

df_conf = pd.DataFrame(confusion_pairs).sort_values('count', ascending=False).head(10)
print('가장 많이 혼동된 클래스 쌍 Top 10:')
print(df_conf.to_string(index=False))

In [ ]:
# ③ 실패 케이스 시각화
# 신뢰도 높게 틀린 케이스 = 모델이 확신을 갖고 틀린 케이스 (가장 중요)
wrong_cases = []
for i in range(len(X_test)):
    true_idx = y_true[i]
    pred_idx = y_pred[i]
    if true_idx != pred_idx:
        wrong_cases.append({
            'img':        X_test[i],
            'true':       class_names[true_idx],
            'pred':       class_names[pred_idx],
            'confidence': y_pred_prob[i][pred_idx]
        })

# 신뢰도 높은 순으로 정렬
wrong_cases.sort(key=lambda x: -x['confidence'])
print(f'총 {len(wrong_cases)}개 오류 케이스')

n = min(12, len(wrong_cases))
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for i in range(n):
    case = wrong_cases[i]
    ax = axes[i//4][i%4]
    ax.imshow(case['img'])
    ax.set_title(
        f"True: {case['true']}\nPred: {case['pred']} ({case['confidence']*100:.1f}%)",
        color='red', fontsize=9
    )
    ax.axis('off')

plt.suptitle('Wrong Predictions with High Confidence (Top 12)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/해커톤/wrong_predictions.png', dpi=100)
plt.show()

In [ ]:
# ④ 실패 원인 분류 (질적 분석)
# CV 모델(best_final.keras)을 streamlit CV 페이지에서 직접 테스트한 결과
# → 실제 서비스 환경에서의 오류 케이스

error_analysis = pd.DataFrame([
    {'원인': 'top_k=3 오분류',  '대표 케이스': 'TV 사진 → laptop 1위',          '개선안': 'top_k=1로 변경'},
    {'원인': '저신뢰도 오분류', '대표 케이스': '페트병 → nightstand (22.3%)',    '개선안': '신뢰도 임계값 설정'},
    {'원인': '형태 유사',       '대표 케이스': '사진(photo) → TV (82.6%)',       '개선안': 'photo 학습 데이터 다양성 확보'},
    {'원인': '유사 카테고리',   '대표 케이스': '소파 → bed (34.0%)',             '개선안': '가구 세부 클래스 데이터 추가'},
    {'원인': '카테고리 오분류', '대표 케이스': '스티로폼 → bed (33.4%)',          '개선안': '신뢰도 임계값 + 데이터 증강'},
])
print(error_analysis.to_string(index=False))

In [ ]:
# ⑤ 클래스별 F1-Score 순위
report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
df_report = pd.DataFrame(report).T[['precision', 'recall', 'f1-score', 'support']].iloc[:-3]
df_report = df_report.sort_values('f1-score', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(df_report.index, df_report['f1-score'], color='coral')
plt.axvline(x=df_report['f1-score'].mean(), color='red', linestyle='--',
            label=f"Mean F1 {df_report['f1-score'].mean():.3f}")
plt.title('Class-wise F1-Score (ascending)')
plt.legend()
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/해커톤/f1_score.png', dpi=100)
plt.show()

print('성능 하위 클래스 (개선 우선순위):')
print(df_report.head(5))

## 오류 분석 결론

단순히 70.5%라는 숫자가 아니라, '왜 29.5%가 틀렸는지'를 분석한 결과
주요 원인은 다음과 같다:

### CV 모델(best_final.keras) 실제 서비스 테스트 결과
1. **top_k=3 설정**: 가장 높은 확률 클래스가 아닌 엉뚱한 클래스가 1위로 나오는 문제
   → top_k=1로 변경하여 해결 가능

2. **형태 유사성으로 인한 오분류**: photo → TV, sofa → bed 등
   → 학습 데이터 다양성 확보 및 증강 필요

3. **저신뢰도 예측**: 신뢰도가 낮을 때도 무조건 분류하는 문제
   → 신뢰도 임계값 설정으로 '분류 불가' 처리 필요

### 비교 실험 데이터셋(roboflow) 분석 결과
4. **fridge 데이터 부족**: train 데이터만 확보되어 valid/test 분리 불가 → F1=0
   → 향후 데이터 추가 수집 필요

5. **album F1=0.4**: 데이터 30장으로 가장 적어 성능 저하
   → 데이터 증강 또는 추가 수집 필요

6. **ring 오분류 多**: chair, air_conditioning 등으로 혼동
   → 천장형 에어컨의 둥근 형태가 반지와 유사하여 혼동 발생

7. **클래스 불균형(8.8배)**: TV 263장 vs album 30장
   → 균형 잡힌 데이터 수집 및 클래스 가중치 적용 필요
   
8. **가전 카테고리 편향**: TV+air_conditioning+fridge 비중 가장 높음
   → 다른 카테고리 예측 성능 저하 가능성

향후 OCR 모듈(EasyOCR)과 결합하면 paper/paper_document 혼동 문제 해결 가능.
실제로 cv_preprocessing.py에 filter_ocr_results 함수가 이미 구현되어 있어 v2에서 통합 예정.